In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda x: '%.2f' % x)

def contar_nulos(df, columna):
    """
    Cuenta y muestra los valores faltantes en una columna 
    """
    total_nulos = df[columna].isnull().sum()
    print(f"Valores faltantes en {columna}: {total_nulos}")
    return total_nulos

def analizar_columna(df, columna):
    """
    Muestra estadísticas descriptivas y conteo de valores para una columna
    """
    print(f"--- Resumen de: {columna} ---")
    print("\n📊 Estadísticas Descriptivas:")
    print(df[columna].describe())
    
    print("\n🔢 Conteo de Valores (Frecuencias):")
    print(df[columna].value_counts())
    print("-" * 30)


## demo_h

### Limpieza Demo_H

In [39]:
df = pd.read_csv("../data/interim/demo_h_columnas.csv")

mapeo_nombres = {
    "RIAGENDR": "Genero",
    "RIDAGEYR": "Edad",
    "RIDRETH3": "Etnia",
    "DMDEDUC2": "Nivel_Educativo",
    "DMDMARTL": "Estado_Civil",
    "INDFMPIR": "Ratio_Pobreza",
    "DMQMILIZ": "Servicio_Militar",
}

df= df.rename(columns=mapeo_nombres)

df = df[df["Edad"] >= 18]

df.head()

### Mapeo de numeros a columnas (opcional) (one-hot enconding)
""" mapeo_etnia = {
    1: "Mexicano_Americano",
    2: "Otro_Hispano",
    3: "Blanco_No_Hispano",
    4: "Negro_No_Hispano",
    6: "Asiatico_No_Hispano",
    7: "Otra_Raza_Multi"
}

df['Etnia'] = df['Etnia'].map(mapeo_etnia)

# mapeo de atributos a columnas True o False
df = pd.get_dummies(df, columns=['Etnia'], prefix='Etnia')

df.head() """

## Limpieza de edad
analizar_columna(df,'Edad')
contar_nulos(df,'Edad')


## Limpieza de Nivel_Educativo
#discrepancia de 344, personas menores a 20 ya que nivel_educativo solo aplica para mayores de 20

analizar_columna(df,'Nivel_Educativo')
contar_nulos(df,'Nivel_Educativo')

def traducir_educacion_juvenil(valor_juvenil):
    if pd.isna(valor_juvenil):
        return np.nan
    
    # Códigos de NHANES para DMDEDUC3 -> DMDEDUC2
    if valor_juvenil in [0, 1, 2, 3, 4, 5, 6, 7, 8, 55, 66]:
        return 1  # Menos de 9no grado
    elif valor_juvenil in [9, 10, 11, 12]:
        return 2  # 9-11 grado (o 12 sin diploma)
    elif valor_juvenil in [13, 14]:
        return 3  # Graduado High School / GED
    elif valor_juvenil == 15:
        return 4  # Algo de universidad
    else:
        return np.nan # Códigos raros (77, 99)

mask_nulos = df['Nivel_Educativo'].isna()
            
df.loc[mask_nulos, 'Nivel_Educativo'] = df.loc[mask_nulos, 'DMDEDUC3'].apply(traducir_educacion_juvenil)

# Filtrado solo por valores validos
df = df[df['Nivel_Educativo'].isin([1, 2, 3, 4, 5])]




## Limpieza de Estado_Civil
# Los menores de edad (NaN) los remplazamos por Nunca Casados.

analizar_columna(df,"Estado_Civil")
contar_nulos(df,"Estado_Civil")

df["Estado_Civil"] = df["Estado_Civil"].replace({77.0: 5, 99.0: 5}).fillna(5)



## Limpieza de Servicio_Militar
# Reemplazar los valores 7 por 2 (No)
contar_nulos(df,"Servicio_Militar")
analizar_columna(df,"Servicio_Militar")

df["Servicio_Militar"] = df["Servicio_Militar"].replace({7: 2})



## Limpieza de Ratio Pobreza
analizar_columna(df,'Ratio_Pobreza')
contar_nulos(df,'Ratio_Pobreza')
#Rellnar los nulos con 0 no es correcto, puesto que los nulos son falta de informacion y 0 tiene un significado real.  
#INDFMPIR: This variable is the ratio of family income to poverty. The Department of Health and Human Services (HHS) poverty guidelines were used as the poverty measure to calculate this ratio. These guidelines are issued each year, in the Federal Register, for determining financial eligibility for certain federal programs, such as Head Start, Supplemental Nutrition Assistance Program (SNAP), Special Supplemental Nutrition Program for Women, Infants, and Children (WIC), and the National School Lunch Program. The poverty guidelines vary by family size and geographic location (with different guidelines for the 48 contiguous states and the District of Columbia; Alaska; and Hawaii).
#INDFMPIR was calculated by dividing family (or individual) income by the poverty guidelines specific to the survey year. The value was not computed if the respondent only reported income as < $20,000 or ≥ $20,000. If family income was reported as a more detailed category, the midpoint of the range was used to compute the ratio. Values at or above 5.00 were coded as 5.00 or more because of disclosure concerns. The values were not computed if the income data was missing.

# Crear una columna bandera (1 si era nulo, 0 si no)
df["Ratio_Pobreza_Faltante"] = df["Ratio_Pobreza"].isnull().astype(int)
# Rellenar el nulo con la mediana
mediana_pobreza = df["Ratio_Pobreza"].median()
df["Ratio_Pobreza"] = df["Ratio_Pobreza"].fillna(mediana_pobreza)




--- Resumen de: Edad ---

📊 Estadísticas Descriptivas:
count   6113.00
mean      47.39
std       18.47
min       18.00
25%       32.00
50%       47.00
75%       62.00
max       80.00
Name: Edad, dtype: float64

🔢 Conteo de Valores (Frecuencias):
Edad
80.00    352
18.00    191
19.00    153
63.00    119
30.00    117
        ... 
76.00     51
75.00     51
77.00     38
78.00     36
79.00     30
Name: count, Length: 63, dtype: int64
------------------------------
Valores faltantes en Edad: 0
--- Resumen de: Nivel_Educativo ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       3.52
std        1.24
min        1.00
25%        3.00
50%        4.00
75%        5.00
max        9.00
Name: Nivel_Educativo, dtype: float64

🔢 Conteo de Valores (Frecuencias):
Nivel_Educativo
4.00    1770
5.00    1443
3.00    1303
2.00     791
1.00     455
9.00       5
7.00       2
Name: count, dtype: int64
------------------------------
Valores faltantes en Nivel_Educativo: 344
--- Resumen de: Estado_Civil ---



In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6106 entries, 0 to 10172
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SEQN                    6106 non-null   float64
 1   Genero                  6106 non-null   float64
 2   Edad                    6106 non-null   float64
 3   Etnia                   6106 non-null   float64
 4   Nivel_Educativo         6106 non-null   float64
 5   DMDEDUC3                344 non-null    float64
 6   Estado_Civil            6106 non-null   float64
 7   Ratio_Pobreza           6106 non-null   float64
 8   Servicio_Militar        6106 non-null   float64
 9   Ratio_Pobreza_Faltante  6106 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 524.7 KB


In [41]:
df = df.drop(columns=['DMDEDUC3'])

df.to_csv('../data/demo.csv', index=False)

## ALQ

### Limpieza alq

In [42]:
df = pd.read_csv("../data/interim/alq_h_columnas.csv")

nuevos_nombres = {
    "ALQ120Q": "Frec_Consumo_Anual",
    "ALQ120U": "Unidad_Frec_Anual",
    "ALQ130":  "Promedio_Tragos_Dia",
    "ALQ141Q": "Dias_4_5_Tragos",
    "ALQ141U": "Unidad_4_5_Tragos", 
    "ALQ160":  "Dias_Con_+4_5_Tragos_En_2hrs"
}

# Aplicar el cambio
df = df.rename(columns=nuevos_nombres)

# Marcar 0s en columnas relevantes para personas que no han consumido alcohol

# Redondear 
df['Frec_Consumo_Anual'] = df['Frec_Consumo_Anual'].round(2)
df['Dias_4_5_Tragos'] = df['Dias_4_5_Tragos'].round(2)
df['Dias_Con_+4_5_Tragos_En_2hrs'] = df['Dias_Con_+4_5_Tragos_En_2hrs'].round(2)


# Creamos una columna si ha consumido alcohol en toda su vida
df['Consumio_Alcohol'] = ((df['ALQ101'] == 1) | (df['ALQ110'] == 1)) & (df['Frec_Consumo_Anual'] > 0)


# Definimos la lista de columnas que queremos poner en 0
columnas_relevantes = [
    "Frec_Consumo_Anual",
    "Unidad_Frec_Anual",
    "Promedio_Tragos_Dia",
    "Dias_4_5_Tragos",
    "Unidad_4_5_Tragos", 
    "Dias_Con_+4_5_Tragos_En_2hrs"
]


# Si 'Consumio_Alcohol' es False, asignamos 0 a todas esas columnas
df.loc[df['Consumio_Alcohol'] == False, columnas_relevantes] = 0

analizar_columna(df,'Consumio_Alcohol')
contar_nulos(df,'Consumio_Alcohol')


# Crearemos una sola columna Dias_Consumo_Anual para unir la frecuencia y el consumo

### Limpieza de Frec_Consumo_Anual, Mantenemos nulos
# Filtramos el dataframe para excluir las filas con 999
df = df[df['Frec_Consumo_Anual'] != 999]

# Reseteamos el índice para mantener el orden
df = df.reset_index(drop=True)
analizar_columna(df,'Frec_Consumo_Anual')
contar_nulos(df,'Frec_Consumo_Anual')



### Limpieza de Unidad_Frec_Anual
analizar_columna(df,'Unidad_Frec_Anual')
contar_nulos(df,'Unidad_Frec_Anual')
# Ya limpio



### Creacion de Dias_Consumo_Anual
factores = {1: 52, 2: 12, 3: 1}
df['Dias_Consumo_Anual'] = df['Frec_Consumo_Anual'] * df['Unidad_Frec_Anual'].map(factores)
# Como los que marcaron 0 no tienen unidad, el resultado da NaN, pero debería ser 0.
df.loc[df['Frec_Consumo_Anual'] == 0, 'Dias_Consumo_Anual'] = 0



### Limpieza Promedio_Tragos Dia

# 2 personas no saben cuanto beben al dia, las eliminaremos por simplicidad
df[df['Promedio_Tragos_Dia']==999]

# Filtramos el dataframe para excluir las filas con 999
df = df[df['Promedio_Tragos_Dia'] != 999]
# Filtramos el dataframe para excluir las filas con nulo
df = df.dropna(subset=['Promedio_Tragos_Dia'])
# Reseteamos el índice para mantener el orden
df = df.reset_index(drop=True)



### Limpieza Dias_4_5_Tragos
df['Dias_4_5_Tragos'] = df['Dias_4_5_Tragos'].round(2)

# Si alguien no bebio, ponemos 0
df.loc[df['Frec_Consumo_Anual'] == 0, 'Dias_4_5_Tragos'] = 0

# dropeamos nulos y codigos invalidos
df = df.dropna(subset=['Dias_4_5_Tragos'])
df = df[df['Dias_4_5_Tragos'] != 999]
df = df.reset_index(drop=True)



# Unidad_4_5_Tragos
# Si no has tomado 4-5 tragos en un dia, marcamos 0 en esta columna
df.loc[df['Dias_4_5_Tragos'] == 0, 'Unidad_4_5_Tragos'] = 0

analizar_columna(df,'Unidad_4_5_Tragos')
contar_nulos(df,'Unidad_4_5_Tragos')


#juntar Dias y Unidad 
factores_consumo = {1: 52, 2: 12, 3: 1}
df['Dias_Alto_Consumo_Anual'] = df['Dias_4_5_Tragos'] * df['Unidad_4_5_Tragos'].map(factores_consumo)

# 3. Manejo de casos específicos:
# Si Dias_4_5_Tragos es 0, el resultado debe ser 0 (evitando NaNs por falta de unidad)
df.loc[df['Dias_4_5_Tragos'] == 0, 'Dias_Alto_Consumo_Anual'] = 0

# 4. Limpieza final de nulos en la nueva columna si fuera necesario
df = df.dropna(subset=['Dias_Alto_Consumo_Anual'])
df = df.reset_index(drop=True)

analizar_columna(df, 'Dias_Alto_Consumo_Anual')
contar_nulos(df, 'Dias_Alto_Consumo_Anual')





# Dias_Con_+4_5_Tragos_En_2hrs
# si no tienes un dia con 4/5 tragos, marcar 0 
df.loc[df['Dias_4_5_Tragos'] == 0, 'Dias_Con_+4_5_Tragos_En_2hrs'] = 0
df = df[df['Dias_Con_+4_5_Tragos_En_2hrs'] != 999]
df = df.reset_index(drop=True)

analizar_columna(df,'Dias_Con_+4_5_Tragos_En_2hrs')
contar_nulos(df,'Dias_Con_+4_5_Tragos_En_2hrs')

--- Resumen de: Consumio_Alcohol ---

📊 Estadísticas Descriptivas:
count     5924
unique       2
top       True
freq      3597
Name: Consumio_Alcohol, dtype: object

🔢 Conteo de Valores (Frecuencias):
Consumio_Alcohol
True     3597
False    2327
Name: count, dtype: int64
------------------------------
Valores faltantes en Consumio_Alcohol: 0
--- Resumen de: Frec_Consumo_Anual ---

📊 Estadísticas Descriptivas:
count   5920.00
mean       2.89
std       15.19
min        0.00
25%        0.00
50%        1.00
75%        3.00
max      365.00
Name: Frec_Consumo_Anual, dtype: float64

🔢 Conteo de Valores (Frecuencias):
Frec_Consumo_Anual
0.00      2327
1.00      1046
2.00       884
3.00       524
4.00       302
5.00       265
7.00       194
6.00       141
10.00       70
20.00       24
12.00       24
8.00        23
15.00       22
30.00       20
14.00        6
365.00       6
9.00         4
25.00        4
24.00        4
40.00        2
80.00        2
11.00        2
16.00        2
100.00       2
28.

np.int64(0)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5909 entries, 0 to 5908
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   SEQN                          5909 non-null   float64
 1   ALQ101                        5406 non-null   float64
 2   ALQ110                        1630 non-null   float64
 3   Frec_Consumo_Anual            5909 non-null   float64
 4   Unidad_Frec_Anual             5909 non-null   float64
 5   Promedio_Tragos_Dia           5909 non-null   float64
 6   Dias_4_5_Tragos               5909 non-null   float64
 7   Unidad_4_5_Tragos             5909 non-null   float64
 8   Dias_Con_+4_5_Tragos_En_2hrs  5909 non-null   float64
 9   Consumio_Alcohol              5909 non-null   bool   
 10  Dias_Consumo_Anual            5909 non-null   float64
 11  Dias_Alto_Consumo_Anual       5909 non-null   float64
dtypes: bool(1), float64(11)
memory usage: 513.7 KB


In [44]:
df = df.drop(columns=['ALQ101','ALQ110','Frec_Consumo_Anual','Unidad_Frec_Anual','Dias_4_5_Tragos','Unidad_4_5_Tragos'])

df.to_csv('../data/alq.csv', index=False)

# mcq


In [45]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: '%.2f' % x)

def contar_nulos(df, columna):
    """
    Cuenta y muestra los valores faltantes en una columna 
    """
    total_nulos = df[columna].isnull().sum()
    print(f"Valores faltantes en {columna}: {total_nulos}")
    return total_nulos

def analizar_columna(df, columna):
    """
    Muestra estadísticas descriptivas y conteo de valores para una columna
    """
    print(f"--- Resumen de: {columna} ---")
    print("\n📊 Estadísticas Descriptivas:")
    print(df[columna].describe())
    
    print("\n🔢 Conteo de Valores (Frecuencias):")
    print(df[columna].value_counts())
    print("-" * 30)

def analizar(df,columna):
    analizar_columna(df, columna)
    contar_nulos(df,columna)

df = pd.read_csv("../data/interim/mcq_h_columnas.csv")
# Filtrar solo mayores a 18
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
df = df.merge(df_edad,on='SEQN',how='inner')
df = df[df['RIDAGEYR']>=18]
renombrar_condiciones_medicas = {
    "MCQ160M": "problema_tiroides",
    "MCQ160L": "problema_higado",
    "MCQ160A": "tiene_artritis",
    "MCQ160F": "tuvo_derrame_cerebral",
    "MCQ160E": "tuvo_ataque_corazon",
    "MCQ160C": "enfermedad_coronaria",
    "MCQ220": "tuvo_cancer",
    "MCQ010": "tiene_asma",
    "MCQ080": "tiene_sobrepeso",
    "MCQ070": "tiene_psoriasis"
}

# Para aplicarlo a tu DataFrame:
df = df.rename(columns=renombrar_condiciones_medicas)
nulos_por_columna = df.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)
cols_medicas = [
    "problema_tiroides",
    "problema_higado",
    "tiene_artritis",
    "tuvo_derrame_cerebral",
    "tuvo_ataque_corazon",
    "enfermedad_coronaria",
    "tuvo_cancer",
    "tiene_asma",
    "tiene_sobrepeso",
    "tiene_psoriasis"
]

for col in cols_medicas:
    analizar(df,col)

# Los 7 (Refused), 9 (Don't Know) y los vacíos originales se vuelven NaN.
for col in cols_medicas:
    df[col] = df[col].map({1: True, 2: False})

# Rellenar todos los NaN con False
# jóvenes de 18-19 años.
df[cols_medicas] = df[cols_medicas].fillna(False)

# 3. Verificamos que ya no queden nulos
print("Nulos restantes en columnas médicas:")
print(df[cols_medicas].isna().sum())
df = df.drop(columns=['RIDAGEYR'])

df.to_csv('../data/mcq.csv', index=False)

Cantidad de nulos por columna:
SEQN                       0
problema_tiroides        344
problema_higado          344
tiene_artritis           344
tuvo_derrame_cerebral    344
tuvo_ataque_corazon      344
enfermedad_coronaria     344
tuvo_cancer              344
tiene_asma                 0
tiene_sobrepeso            0
tiene_psoriasis            0
RIDAGEYR                   0
dtype: int64
--- Resumen de: problema_tiroides ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       1.91
std        0.44
min        1.00
25%        2.00
50%        2.00
75%        2.00
max        9.00
Name: problema_tiroides, dtype: float64

🔢 Conteo de Valores (Frecuencias):
problema_tiroides
2.00    5157
1.00     601
9.00      11
Name: count, dtype: int64
------------------------------
Valores faltantes en problema_tiroides: 344
--- Resumen de: problema_higado ---

📊 Estadísticas Descriptivas:
count   5769.00
mean       1.97
std        0.35
min        1.00
25%        2.00
50%        2.00
75%        2.00


/tmp/ipykernel_276530/4182567343.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols_medicas] = df[cols_medicas].fillna(False)


# paq

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: '%.2f' % x)

def contar_nulos(df, columna):
    """
    Cuenta y muestra los valores faltantes en una columna 
    """
    total_nulos = df[columna].isnull().sum()
    print(f"Valores faltantes en {columna}: {total_nulos}")
    return total_nulos

def analizar_columna(df, columna):
    """
    Muestra estadísticas descriptivas y conteo de valores para una columna
    """
    print(f"--- Resumen de: {columna} ---")
    print("\n📊 Estadísticas Descriptivas:")
    print(df[columna].describe())
    
    print("\n🔢 Conteo de Valores (Frecuencias):")
    print(df[columna].value_counts())
    print("-" * 30)

def analizar(df,columna):
    analizar_columna(df, columna)
    contar_nulos(df,columna)

# Limpieza PAQ

df = pd.read_csv("../data/interim/paq_h_columnas.csv")

# Diccionario de mapeo
renombrar_actividad = {
    "PAD680": "minutos_sedentario_dia",
    "PAQ650": "ejercicio_vigoroso_recreativo",
    "PAQ665": "ejercicio_moderado_recreativo",
    "PAQ605": "trabajo_vigoroso",
    "PAQ620": "trabajo_moderado",
    "PAQ635": "transporte_activo_bici_caminar"
}
df = df.rename(columns=renombrar_actividad)


df
# Filtrar solo mayores a 18
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
df = df.merge(df_edad,on='SEQN',how='inner')
df = df[df['RIDAGEYR']>=18]

nulos_por_columna = df.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)
analizar_columna(df,"minutos_sedentario_dia")
contar_nulos(df,'minutos_sedentario_dia')
df['minutos_sedentario_dia'] = df['minutos_sedentario_dia'].replace([7777, 9999], np.nan)
df = df.dropna(subset=['minutos_sedentario_dia'])
# 3. Conversión de minutos a horas
df['horas_sedentario_dia'] = df['minutos_sedentario_dia'] / 60
# Reseteamos el índice para mantener el orden
df = df.reset_index(drop=True)
### ejercicio_vigoroso_recreativo
analizar(df,'ejercicio_vigoroso_recreativo')
# Definimos el mapeo
mapa_logico = {1: True, 2: False}

# Aplicamos el mapeo a la columna
df['ejercicio_vigoroso_recreativo'] = df['ejercicio_vigoroso_recreativo'].map(mapa_logico)

# Verificamos los resultados
print(df['ejercicio_vigoroso_recreativo'].value_counts(dropna=False))
### .ejercicio_moderado_recreativo
analizar(df,'ejercicio_moderado_recreativo')
# Definimos el mapeo
mapa_logico = {1: True, 2: False}

# Aplicamos el mapeo a la columna
df['ejercicio_moderado_recreativo'] = df['ejercicio_moderado_recreativo'].map(mapa_logico)

# Verificamos los resultados
print(df['ejercicio_moderado_recreativo'].value_counts(dropna=False))
### trabajo_vigoroso
analizar(df,'trabajo_vigoroso')
# Definimos el mapeo
mapa_logico = {1: True, 2: False}

# Aplicamos el mapeo a la columna
df['trabajo_vigoroso'] = df['trabajo_vigoroso'].map(mapa_logico)

# Verificamos los resultados
print(df['trabajo_vigoroso'].value_counts(dropna=False))
df = df.dropna(subset=['trabajo_vigoroso'])

### trabajo_moderado
analizar(df,'trabajo_moderado')
# Definimos el mapeo
mapa_logico = {1: True, 2: False}

# Aplicamos el mapeo a la columna
df['trabajo_moderado'] = df['trabajo_moderado'].map(mapa_logico)

# Verificamos los resultados

print(df['trabajo_moderado'].value_counts(dropna=False))
df = df.dropna(subset=['trabajo_moderado'])
print(df['trabajo_moderado'].value_counts(dropna=False))

### transporte_activo_bici_caminar
analizar(df,'transporte_activo_bici_caminar')
# Definimos el mapeo
mapa_logico = {1: True, 2: False}

# Aplicamos el mapeo a la columna
df['transporte_activo_bici_caminar'] = df['transporte_activo_bici_caminar'].map(mapa_logico)

# Verificamos los resultados  (eliminar nulos)
print(df['transporte_activo_bici_caminar'].value_counts(dropna=True))
df = df.dropna(subset=['trabajo_moderado'])

df = df.drop(columns=['RIDAGEYR'])
df.to_csv('../data/paq.csv', index=False)

Cantidad de nulos por columna:
SEQN                              0
minutos_sedentario_dia            5
ejercicio_vigoroso_recreativo     0
ejercicio_moderado_recreativo     0
trabajo_vigoroso                  0
trabajo_moderado                  0
transporte_activo_bici_caminar    0
RIDAGEYR                          0
dtype: int64
--- Resumen de: minutos_sedentario_dia ---

📊 Estadísticas Descriptivas:
count   6108.00
mean     454.09
std      589.11
min        0.00
25%      240.00
50%      480.00
75%      540.00
max     9999.00
Name: minutos_sedentario_dia, dtype: float64

🔢 Conteo de Valores (Frecuencias):
minutos_sedentario_dia
480.00     1355
600.00      664
360.00      639
240.00      603
300.00      563
180.00      424
720.00      372
120.00      369
540.00      299
420.00      284
60.00       115
660.00       95
840.00       86
900.00       57
960.00       46
780.00       45
9999.00      20
1080.00      17
30.00        13
90.00        11
1020.00       7
20.00         4
15.00      

# whq

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: '%.2f' % x)

def contar_nulos(df, columna):
    """
    Cuenta y muestra los valores faltantes en una columna 
    """
    total_nulos = df[columna].isnull().sum()
    print(f"Valores faltantes en {columna}: {total_nulos}")
    return total_nulos

def analizar_columna(df, columna):
    """
    Muestra estadísticas descriptivas y conteo de valores para una columna
    """
    print(f"--- Resumen de: {columna} ---")
    print("\n📊 Estadísticas Descriptivas:")
    print(df[columna].describe())
    
    print("\n🔢 Conteo de Valores (Frecuencias):")
    print(df[columna].value_counts())
    print("-" * 30)


df = pd.read_csv("../data/interim/whq_h_columnas.csv")

def analizar(columna,df=df):
    analizar_columna(df, columna)
    contar_nulos(df,columna)
# Filtrar solo mayores a 18
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
df = df.merge(df_edad,on='SEQN',how='inner')
df = df[df['RIDAGEYR']>=18]
nulos_por_columna = df.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)
renombrar_peso_historia = {
    "WHD010": "altura_pulgadas",
    "WHD020": "peso_libras",
    "WHQ030": "percibe_peso_como", # 1:Sobrepeso, 2:Bajo, 3:Bien
    "WHQ060": "perdio_peso_intencional",
    "WHQ070": "intento_bajar_peso_1año",
    "WHD080E": "conducta_saltar_comidas",
    "WHD080K": "conducta_laxantes_vomito",
    "WHD080P": "conducta_fumar_para_adelgazar"
}

df = df.rename(columns=renombrar_peso_historia)
## Altura en pulgadas
analizar('altura_pulgadas',df)

### whd010 altura

# remover 7777 y 9999 y nulos
df['altura_pulgadas'] = df['altura_pulgadas'].replace([7777, 9999], np.nan)

df = df.dropna(subset=['altura_pulgadas'])

df['altura_cm'] = df['altura_pulgadas'] * 2.54
## Peso libras
analizar('peso_libras',df)
# peso_libras
df.loc[:, 'peso_libras'] = df['peso_libras'].replace([7777, 9999, 77777, 99999], np.nan)
df = df.dropna(subset=['peso_libras'])
analizar('peso_libras',df)
## Percibe peso
# percibe peso
mapa_percepcion = {
    1: 'Sobrepeso',
    2: 'Bajo peso',
    3: 'Peso ideal'
}

# 2. Aplicamos el mapeo con .map()
df['percibe_peso_como'] = df['percibe_peso_como'].map(mapa_percepcion)

# 
df = df.dropna(subset=['percibe_peso_como'])

analizar('percibe_peso_como',df)
## perdio_peso_intencional
df['perdio_peso_intencional'] = df['perdio_peso_intencional'].map({1: True, 2: False})

analizar('perdio_peso_intencional',df)
## intento_bajar_peso_1año
analizar('intento_bajar_peso_1año',df)
df['intento_bajar_peso_1año'] = df['intento_bajar_peso_1año'].replace(9, np.nan)
df['intento_bajar_peso_1año'] = df['intento_bajar_peso_1año'].map({1: True, 2: False})
analizar('intento_bajar_peso_1año',df)

## nueva col: intento_bajar_peso

df['intento_bajar_peso'] = False
df.loc[df['perdio_peso_intencional'] == 1, 'intento_bajar_peso'] = True
df.loc[df['intento_bajar_peso_1año'] == 1, 'intento_bajar_peso'] = True

analizar('intento_bajar_peso',df)

nulos_por_columna = df.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)

# 1. Definimos las columnas que queremos unir
columnas_riesgo = [
    'conducta_saltar_comidas',
    'conducta_laxantes_vomito', 
    'conducta_fumar_para_adelgazar'
]

# 2. Creamos la nueva columna
# .any(axis=1) revisa fila por fila. Si encuentra al menos un True, el resultado es True.
df['conductas_riesgo_peso'] = df[columnas_riesgo].any(axis=1)

# 3. (Opcional pero recomendado) Borramos las 3 columnas originales para no tener datos duplicados
df = df.drop(columns=columnas_riesgo)

# 4. Verificamos cuántos casos positivos logramos juntar
print(df['conductas_riesgo_peso'].value_counts())
df = df.drop(columns=['RIDAGEYR', 'perdio_peso_intencional', 'intento_bajar_peso_1año',])

df.to_csv('../data/whq.csv', index=False)
df.info()

Cantidad de nulos por columna:
SEQN           0
WHD010        24
WHD020        28
WHQ030         0
WHQ060      4917
WHQ070       762
WHD080E     5741
WHD080K     6089
WHD080P     6094
RIDAGEYR       0
dtype: int64
--- Resumen de: altura_pulgadas ---

📊 Estadísticas Descriptivas:
count   6089.00
mean     168.75
std     1002.01
min       48.00
25%       63.00
50%       66.00
75%       69.00
max     9999.00
Name: altura_pulgadas, dtype: float64

🔢 Conteo de Valores (Frecuencias):
altura_pulgadas
66.00      599
64.00      530
65.00      509
67.00      476
62.00      469
63.00      467
68.00      435
69.00      420
70.00      375
71.00      328
72.00      271
61.00      253
60.00      230
73.00      170
74.00      150
59.00      107
75.00       64
9999.00     62
76.00       52
58.00       37
57.00       26
77.00       18
56.00       14
55.00        9
78.00        5
54.00        4
79.00        3
7777.00      1
81.00        1
48.00        1
49.00        1
80.00        1
53.00        1
Name: c

# dpq


In [4]:
df_dpq = pd.read_csv('../data/interim/dpq_h_columnas.csv')
df_dpq.info()
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
df_dpq = df_dpq.merge(df_edad,on='SEQN',how='inner')
df_dpq = df_dpq[df_dpq['RIDAGEYR']>=18]
nulos_por_columna = df_dpq.isna().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)
# Lista de columnas del cuestionario
cols_dpq = ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']

# Verificar cuántos sujetos tienen nulos en TODO el cuestionario
nulos_totales = df_dpq[cols_dpq].isna().all(axis=1).sum()
print(f"Usuarios sin ninguna respuesta: {nulos_totales}")




# 2. Definir las columnas de los síntomas (dpq010 hasta dpq090)
columnas_sintomas = [f'DPQ0{i}0' for i in range(1, 10)]

# 3. Reemplazar los códigos 7 (Refused) y 9 (Don't know) por NaN
df_dpq[columnas_sintomas] = df_dpq[columnas_sintomas].replace({7: np.nan, 9: np.nan})

# Eliminar filas donde las respuestas del PHQ-9 sean nulas
df_dpq = df_dpq.dropna(subset=cols_dpq, how='any')

# 4. Crear el Score Continuo (suma de 0 a 27)
# Usamos skipna=False para que si a un paciente le falta una respuesta, su total sea NaN
# (Es más seguro para no sesgar el modelo con sumas incompletas)
# 1. Crear el Score Continuo (suma de 0 a 27)
df_dpq['score_depresion'] = df_dpq[columnas_sintomas].sum(axis=1, skipna=False)

# 2. ELIMINAR los nulos generados por los 7s y 9s (las 21 filas)
# Hacemos esto ANTES de crear df_final
df_dpq = df_dpq.dropna(subset=['score_depresion'])

# 3. Seleccionar solo las columnas deseadas ahora que está limpio
df_final = df_dpq[['SEQN', 'score_depresion']]

# Ver el resultado
print(df_final.info())
df_final.to_csv('../data/processed/dpq_limpio.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5924 entries, 0 to 5923
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   SEQN    5924 non-null   float64
 1   DPQ010  5398 non-null   float64
 2   DPQ020  5396 non-null   float64
 3   DPQ030  5395 non-null   float64
 4   DPQ040  5395 non-null   float64
 5   DPQ050  5395 non-null   float64
 6   DPQ060  5394 non-null   float64
 7   DPQ070  5394 non-null   float64
 8   DPQ080  5394 non-null   float64
 9   DPQ090  5393 non-null   float64
 10  DPQ100  3674 non-null   float64
dtypes: float64(11)
memory usage: 509.2 KB
Cantidad de nulos por columna:
SEQN           0
DPQ010       526
DPQ020       528
DPQ030       529
DPQ040       529
DPQ050       529
DPQ060       530
DPQ070       530
DPQ080       530
DPQ090       531
DPQ100      2250
RIDAGEYR       0
dtype: int64
Usuarios sin ninguna respuesta: 526
<class 'pandas.core.frame.DataFrame'>
Index: 5372 entries, 0 to 5922
Data columns (to